# SONA MS-SNSD GTCRN External Validation

Evaluates the best loss-focused VoiceBank GTCRN checkpoint on the full MS-SNSD external validation set. This is a branch result; locked strict guarded attention remains the safe main baseline unless GTCRN wins externally.

In [ ]:
%pip install -q pystoi pesq ptflops

import glob
import os
import sys
from pathlib import Path

IN_KAGGLE = Path('/kaggle/working').exists()
print('IN_KAGGLE =', IN_KAGGLE)

In [ ]:
def find_repo():
    here = Path.cwd()
    if (here / 'scripts' / 'eval_external_ms_snsd.py').exists():
        return str(here)
    hits = glob.glob('/kaggle/input/**/scripts/eval_external_ms_snsd.py', recursive=True)
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1]) if hits else None

def find_manifest():
    local = r'D:\noise cancellation\external_data\ms_snsd_sona\test.csv'
    if Path(local).exists():
        return local
    hits = [p for p in glob.glob('/kaggle/input/**/test.csv', recursive=True) if 'ms' in p.lower() and 'snsd' in p.lower()]
    return sorted(hits, key=len)[0] if hits else None

def find_checkpoint():
    local = r'D:\noise cancellation\results\kaggle_voicebank_gtcrn_loss_focus_full824_2026-07-06\extracted\ckpt_gtcrn_ch96_h96_b4_mw1p25_mr0p25\last.pt'
    if Path(local).exists():
        return local
    hits = glob.glob('/kaggle/input/**/ckpt_gtcrn_ch96_h96_b4_mw1p25_mr0p25/last.pt', recursive=True)
    if not hits:
        hits = [p for p in glob.glob('/kaggle/input/**/last.pt', recursive=True) if 'gtcrn' in p.lower() and 'mw1p25' in p.lower() and 'mr0p25' in p.lower()]
    return sorted(hits, key=len)[0] if hits else None

REPO_DIR = find_repo()
MANIFEST = find_manifest()
CHECKPOINT = find_checkpoint()
OUT_DIR = '/kaggle/working/ms_snsd_gtcrn_external_eval' if IN_KAGGLE else r'D:\noise cancellation\results\ms_snsd_gtcrn_external_eval'

# If auto-detect fails, set manually from the Kaggle right panel:
# REPO_DIR = '/kaggle/input/sona-new'
# MANIFEST = '/kaggle/input/sona-ms-snsd-external-dataset/ms_snsd_sona/test.csv'
# CHECKPOINT = '/kaggle/input/sona-voicebank-gtcrn-loss-focus-full824/ckpt_gtcrn_ch96_h96_b4_mw1p25_mr0p25/last.pt'

assert REPO_DIR, 'Repo with scripts/eval_external_ms_snsd.py not found.'
assert MANIFEST, 'MS-SNSD manifest test.csv not found.'
assert CHECKPOINT, 'GTCRN checkpoint last.pt not found.'

print('REPO_DIR   =', REPO_DIR)
print('MANIFEST   =', MANIFEST)
print('CHECKPOINT =', CHECKPOINT)
print('OUT_DIR    =', OUT_DIR)

In [ ]:
# 0 = full MS-SNSD external test set.
LIMIT = 0
METHOD_NAME = 'gtcrn_ch96_h96_b4_mw1p25_mr0p25'
ADAPTER = 'lowrank'
ANCHORS = '2.0'
STEPS = '1'
MAX_GAIN_DELTAS = '0.01'
GUARD_MIN_REWARD_DELTAS = 'none'
GUARD_MAX_OUTPUT_DELTAS = '0.0001'
GUARD_MIN_MASK_POLARITIES = 'none'
REWARD_KIND = 'auto'

print(dict(
    LIMIT=LIMIT,
    METHOD_NAME=METHOD_NAME,
    ADAPTER=ADAPTER,
    ANCHORS=ANCHORS,
    STEPS=STEPS,
    MAX_GAIN_DELTAS=MAX_GAIN_DELTAS,
    GUARD_MAX_OUTPUT_DELTAS=GUARD_MAX_OUTPUT_DELTAS,
    REWARD_KIND=REWARD_KIND,
))

In [ ]:
import subprocess

cmd = [
    sys.executable, str(Path(REPO_DIR) / 'scripts' / 'eval_external_ms_snsd.py'),
    '--repo', REPO_DIR,
    '--manifest', MANIFEST,
    '--checkpoint', CHECKPOINT,
    '--out', OUT_DIR,
    '--method-name', METHOD_NAME,
    '--limit', str(LIMIT),
    '--adapter', ADAPTER,
    '--anchors', ANCHORS,
    '--steps', STEPS,
    '--max-gain-deltas', MAX_GAIN_DELTAS,
    '--guard-min-reward-deltas', GUARD_MIN_REWARD_DELTAS,
    '--guard-max-output-deltas', GUARD_MAX_OUTPUT_DELTAS,
    '--guard-min-mask-polarities', GUARD_MIN_MASK_POLARITIES,
    '--reward-kind', REWARD_KIND,
]

print(' '.join(cmd), '\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'eval_external_ms_snsd.py exited {proc.returncode}'

In [ ]:
import pandas as pd

summary_md = Path(OUT_DIR) / 'summary.md'
summary_csv = Path(OUT_DIR) / 'summary.csv'
print(summary_md.read_text(encoding='utf-8'))
df = pd.read_csv(summary_csv)
display(df.sort_values(['pesq', 'si_sdr_i'], ascending=[False, False]))

base = df[df.method == 'neural_' + METHOD_NAME].iloc[0]
online = df[df.method.str.startswith(METHOD_NAME + '_online_')].copy()
for col in ['si_sdr_i', 'pesq', 'stoi', 'estoi', 'lsd']:
    online['delta_' + col] = online[col] - base[col]
display(online.sort_values(['delta_pesq', 'delta_si_sdr_i'], ascending=[False, False]))

In [ ]:
import shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_ms_snsd_gtcrn_external_eval' if IN_KAGGLE else r'D:\noise cancellation\results\sona_ms_snsd_gtcrn_external_eval'
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip") / 1e6:.1f} MB)')
display(FileLink(ART + '.zip'))